# 01 - data exploration
carrefour data challenge — initial data loading and exploration.

## one-time setup: convert raw csv to parquet
run this cell once after placing the raw csv files in `data/raw/csv/`. safe to re-run — already-converted files are skipped.

In [11]:
import sys
sys.path.append('..')

from src.data_loader import convert_csv_to_parquet

convert_csv_to_parquet()

maestra_articulos: already converted, skipping
linea_tickets: already converted, skipping


## load data

In [12]:
from src.data_loader import load_maestra_articulos, load_linea_tickets

df_articulos = load_maestra_articulos()
df_tickets = load_linea_tickets()

print("articulos:", df_articulos.shape)
print("tickets:  ", df_tickets.shape)

articulos: (893944, 4)
tickets:   (191017715, 10)


## maestra_articulos
product catalogue — 893k articles across sectors.

In [13]:
# schema and null check
df_articulos.dtypes.to_frame("dtype").assign(nulls=df_articulos.isnull().sum())

,dtype,nulls
idarticu,int64,0
desc_larga_articulo,str,0
idsector,int64,0
desc_sector,str,0


In [14]:
# products per sector
df_articulos.groupby(["idsector", "desc_sector"]).size().rename("n_products").sort_values(ascending=False)

idsector  desc_sector         
3         BAZAR                   425050
1         P.G.C.                  171672
6         TEXTIL                  132735
2         PROD. FRESCOS TRADIC     88033
4         ELECTROFOTO              75979
7         GASOLINERA                 475
Name: n_products, dtype: int64

In [15]:
df_articulos.head()

,idarticu,desc_larga_articulo,idsector,desc_sector
0,966557,GUISO DE VERDURAS LITORAL 415 GR,1,P.G.C.
1,909163,LA HISTORIA DE LOS 3 CERDITOS ANAYA,3,BAZAR
2,594029,MUU (SUSAETA),3,BAZAR
3,86412,ALAS DE MOSCA PARA ÃNGEL ANAYA EDUCACIÃN,3,BAZAR
4,507503,CAMISETA NINO MANGA LARGA SPIDERMAN TINTA META...,6,TEXTIL


## linea_tickets
191m transaction lines — the core input for the pipeline.

In [19]:
df_tickets.head()

,idempres,fecha,hora,ticket,cliente,idarticu,unidades,importe,idpromoc,idtiprod
0,7,2022-01-27,1921,2022-01-2700250025022004292,f228d04042276bdce9881998562b10db86e0d18842008d...,819748,2,3.64,Promo,2
1,7,2022-06-20,2046,2022-06-2000500050035008680,b0b59dac05c99607453446fbc83e9ee316ce56a5bde5b9...,507244,1,2.10,No promo,1
2,7,2022-05-19,1824,2022-05-1900990099005008383,7a253da3aceb4cedd07c3b0b7d9e6beb4718e6bb6aaf4d...,649683,1,1.45,No promo,1
3,7,2022-03-26,1146,2022-03-2600700070019007694,f98195913fef2b5b4a4f95422cce51edfbe05453cc84b6...,65358,1,2.61,Promo,2
4,2,2022-02-13,1851,2022-02-1368186818154003632,37b896c5d0c3bcb141ff0bb17ee92435770ab579ca5174...,261966,1,1.01,No promo,1


In [16]:
# schema and null check
df_tickets.dtypes.to_frame("dtype").assign(nulls=df_tickets.isnull().sum())

,dtype,nulls
idempres,int64,0
fecha,object,0
hora,int64,0
ticket,str,0
cliente,str,0
idarticu,int64,0
unidades,int64,0
importe,float64,0
idpromoc,str,0
idtiprod,int64,0


In [17]:
# scale of the dataset — may take ~30s on 191m rows
print(f"unique customers : {df_tickets['cliente'].nunique():>12,}")
print(f"unique products  : {df_tickets['idarticu'].nunique():>12,}")
print(f"unique tickets   : {df_tickets['ticket'].nunique():>12,}")
print(f"date range       :  {df_tickets['fecha'].min()} → {df_tickets['fecha'].max()}")
print(f"stores (idempres): {df_tickets['idempres'].nunique():>12,}")

unique customers :    1,482,715
unique products  :      117,701
unique tickets   :   20,026,724
date range       :  2022-01-01 → 2022-06-30
stores (idempres):            4


In [18]:
# promotional vs non-promotional lines
df_tickets["idpromoc"].value_counts(normalize=True).mul(100).round(1).rename("share (%)")

idpromoc
No promo    77.6
Promo       22.4
Name: share (%), dtype: float64